<a href="https://colab.research.google.com/github/prodigal94/food-safe-bots/blob/main/Week4_LLR_FineTuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import log

from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.regression import LinearRegression
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import RegressionEvaluator

import matplotlib.pyplot as plt

spark = SparkSession.builder \
    .appName("Week_4_Model_Fine_Tuning") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
    .config("spark.driver.memory", "8g") \
    .config("spark.executor.memory", "8g") \
    .getOrCreate()

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
df = spark.read.parquet("/content/drive/MyDrive/BigData Final Project Files/Parquet/02_AgriTrade_ValueOnly.parquet")
df = df.withColumn("logValue", log(df["Value"]))

In [26]:
from pyspark.sql.functions import udf, col
from pyspark.sql.types import FloatType

flagWeight = {"A":1.0, "X":1.0, "E":.7, "I":.3}
def returnWeight(x):
    return flagWeight[x]
weight_udf = udf(returnWeight, FloatType())

df = df.withColumn("Weight", weight_udf(col("Flag")))

In [27]:
year_count_df = df.groupBy("Year").count().orderBy("Year")

total_records = df.count()
target_count = total_records * 0.8
split_year = None
cumulative_count = 0

# Find the year with a split closest to 80:20
for row in year_count_df.collect():
    cumulative_count += row['count']
    diff = abs(target_count - cumulative_count)
    if cumulative_count >= target_count:
        split_year = row['Year'] if diff > abs(cumulative_count - target_count) else row['Year'] - 1
        break

# Incase no split year was found
if split_year is None:
    print("Warning: Didn't find an 80-20 split. Using last year as split_year.")
    split_year = year_count_df.agg(F.max("Year")).collect()[0][0]

train = df.filter(F.col("Year") <= split_year)
test = df.filter(F.col("Year") > split_year)

print(f"Train dataset size: {train.count()}")
print(f"Test dataset size: {test.count()}")

Train dataset size: 3421072
Test dataset size: 894329


In [28]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml import Pipeline

country_indexer = StringIndexer(inputCol="Area", outputCol="Area_index", handleInvalid="keep")
item_indexer = StringIndexer(inputCol="Item", outputCol="Item_index", handleInvalid="keep")

encoder = OneHotEncoder(
    inputCols=["Area_index", "Item_index"],
    outputCols=["Area_vec", "Item_vec"]
)

assembler = VectorAssembler(
    inputCols=["Year", "Area_vec", "Item_vec"],
    outputCol="features"
)

lrModel = LinearRegression(
    featuresCol="features",
    labelCol="logValue",
    weightCol="Weight"
)
pipeline = Pipeline(stages=[
    country_indexer,
    item_indexer,
    encoder,
    assembler,
    lrModel
])

model = pipeline.fit(train)

In [29]:
lr_model = model.stages[-1]
training_summary = lr_model.summary

print(f"RMSE: {training_summary.rootMeanSquaredError}")
print(f"R2: {training_summary.r2}")

RMSE: 2.355250488165182
R2: 0.5243578743873314


In [30]:
model_path = "/content/drive/MyDrive/BigData Final Project Files/Models/Obj1_LLR_02_BaselineAltWeight"
model.write().overwrite().save(model_path)

In [35]:
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator

lr = [stage for stage in pipeline.getStages() if isinstance(stage, LinearRegression)][0]

# ParamGrid for testing multiple hyperparams
paramGrid = ParamGridBuilder() \
    .addGrid(lr.regParam, [0.01, 0.1, 0.5]) \
    .addGrid(lr.elasticNetParam, [0.0, 0.5, 1.0]) \
    .addGrid(lr.maxIter, [10, 50, 100]) \
    .build()

evaluator = RegressionEvaluator(labelCol="logValue", predictionCol="prediction", metricName="rmse")

# CrossValidator for finding the best model
cv = CrossValidator(
    estimator=pipeline,
    estimatorParamMaps=paramGrid,
    evaluator=evaluator,
    numFolds=3,
    seed=42
)

cvModel = cv.fit(train)

In [38]:
best_lr_model = cvModel.bestModel.stages[-1]

print("Best Linear Regression Model Parameters:")
print(f"  regParam: {best_lr_model.getRegParam()}")
print(f"  elasticNetParam: {best_lr_model.getElasticNetParam()}")
print(f"  maxIter: {best_lr_model.getMaxIter()}")

Best Linear Regression Model Parameters:
  regParam: 0.01
  elasticNetParam: 0.0
  maxIter: 10


In [39]:
predictions = cvModel.transform(test)

# Evaluate the best model on the test set using R2
evaluator_r2 = RegressionEvaluator(labelCol="logValue", predictionCol="prediction", metricName="r2")
r2 = evaluator_r2.evaluate(predictions)
print(f"Test R2 for Best Model (Cross-Validated): {r2}")

Test R2 for Best Model (Cross-Validated): 0.5281518698292818


In [40]:
model_path = "/content/drive/MyDrive/BigData Final Project Files/Models/Obj1_LLR_03_FineTuned"
model.write().overwrite().save(model_path)